In [ ]:
# 安装数据处理和可视化依赖库
import subprocess
import sys

def install_dependencies():
    required_packages = [
        "pandas",    # 数据处理核心库
        "numpy",     # 数值计算库
        "openpyxl",  # Excel 文件读写库（处理.xlsx格式）
        "matplotlib" # 可视化库
    ]
    
    for package in required_packages:
        try:
            __import__(package)
            print(f"✅ {package} 已安装")
        except ImportError:
            print(f"🔄 正在安装 {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"✅ {package} 安装完成")

# 执行依赖安装
install_dependencies()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------
# 1. 设置全局参数（解决中文显示问题）
# --------------------------
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示异常

# --------------------------
# 2. 读取数据（需修改为你的文件路径）
# --------------------------
def load_and_diagnose_data(file_path):
    """
    读取数据并执行初始诊断
    参数: file_path - Excel文件路径（如 "C:/data/小说月票榜.xlsx"）
    返回: 原始数据DataFrame
    """
    # 读取Excel文件（使用openpyxl引擎支持.xlsx）
    try:
        df_original = pd.read_excel(file_path, engine='openpyxl')
        print(f"📊 数据读取成功！原始数据规模：{df_original.shape[0]} 行 × {df_original.shape[1]} 列")
    except Exception as e:
        print(f"❌ 数据读取失败：{str(e)}")
        print("💡 请检查：1.文件路径是否正确 2.文件是否被其他程序占用 3.是否为.xlsx格式")
        return None
    
    # --------------------------
    # 3. 核心问题诊断
    # --------------------------
    print("\n" + "="*60)
    print("🔍 数据初始诊断报告")
    print("="*60)
    
    # 3.1 列名与数据类型
    print("1. 列结构与数据类型：")
    for col in df_original.columns:
        print(f"   - {col}: {df_original[col].dtype}（非空值：{df_original[col].notna().sum()} 个）")
    
    # 3.2 重复值统计
    total_duplicates = df_original.duplicated().sum()  # 完全重复行
    duplicate_rate = (total_duplicates / len(df_original)) * 100
    print(f"\n2. 重复值情况：")
    print(f"   - 完全重复行数：{total_duplicates} 行")
    print(f"   - 重复率：{duplicate_rate:.2f}%")
    
    # 3.3 缺失值统计
    missing_stats = df_original.isnull().sum()
    missing_cols = missing_stats[missing_stats > 0]
    print(f"\n3. 缺失值情况：")
    if len(missing_cols) > 0:
        for col, count in missing_cols.items():
            print(f"   - {col}：{count} 个缺失值（占比：{count/len(df_original)*100:.2f}%）")
    else:
        print(f"   - 无缺失值 ✅")
    
    # 3.4 关键列唯一性（标题、链接等）
    key_cols = ['标题', '标题链接', '作者']  # 根据实际列名调整
    print(f"\n4. 关键列唯一性：")
    for col in key_cols:
        if col in df_original.columns:
            unique_rate = (df_original[col].nunique() / len(df_original)) * 100
            print(f"   - {col}：{df_original[col].nunique()} 个唯一值（唯一性：{unique_rate:.2f}%）")
    
    return df_original

# --------------------------
# 执行数据诊断（修改为你的文件路径！）
# --------------------------
# 示例路径：Windows系统（"C:/用户/文档/小说月票榜.xlsx"）；Mac/Linux系统（"/home/用户/文档/小说月票榜.xlsx"）
file_path = "D :/study/Media data proc/月票榜_小说月票排行—云起书院官网未清洗去查重版.xlsx"  # 🔴 必须修改为实际路径
df_original = load_and_diagnose_data(file_path)

In [ ]:
def export_cleaned_data(df_clean, output_filename="小说月票榜_清洗后数据.xlsx"):
    """
    导出清洗后的数据到Excel文件
    参数: df_clean - 清洗后数据；output_filename - 输出文件名
    返回: 输出文件路径
    """
    try:
        # 使用openpyxl引擎保存.xlsx文件（支持格式优化）
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            df_clean.to_excel(writer, sheet_name='清洗后数据', index=False)
            
            # 优化Excel格式（调整列宽，便于查看）
            worksheet = writer.sheets['清洗后数据']
            column_widths = {
                '标题': 25,      # 标题列宽
                '标题链接': 45,  # 链接列宽（较长）
                'free': 8,       # 会员状态列宽
                '简介': 50,      # 简介列宽（较长）
                '作者': 12,      # 作者列宽
                'rect': 8,       # 分类列宽
                'rect1': 8,      # 状态列宽
                'rect2': 12      # 字数列宽
            }
            
            # 应用列宽设置（匹配实际列名）
            for col_name, width in column_widths.items():
                if col_name in df_clean.columns:
                    # 获取列的字母索引（如A、B、C...）
                    col_idx = df_clean.columns.get_loc(col_name) + 1  # +1是因为Excel列从1开始
                    col_letter = chr(64 + col_idx)  # 64是'A'的ASCII码前一位
                    worksheet.column_dimensions[col_letter].width = width
        
        print(f"\n" + "="*60)
        print("💾 数据导出完成")
        print("="*60)
        print(f"   - 输出文件：{output_filename}")
        print(f"   - 数据规模：{df_clean.shape[0]} 行 × {df_clean.shape[1]} 列")
        print(f"   - 保存位置：当前工作目录（可在VSCode左侧'资源管理器'中查看）")
        return output_filename
    
    except Exception as e:
        print(f"❌ 数据导出失败：{str(e)}")
        print("💡 请检查：1.文件是否被占用 2.当前目录是否有写入权限")
        return None

# --------------------------
# 执行数据导出
# --------------------------
if 'df_clean' in locals() and df_clean is not None:
    export_file = export_cleaned_data(df_clean)
else:
    print("❌ 清洗后数据未生成，无法导出")